In [0]:
IRANIAN_WEB_LOGS_SINGLE_PARQUET = "/Volumes/workspace/default/bronze/iranian_logs-single-parquet"
IRANIAN_WEB_LOGS_PARTITION_PARQUET = "/Volumes/workspace/default/bronze/iranian_logs-partitions-parquet"

In [0]:
df = spark.read.parquet(IRANIAN_WEB_LOGS_SINGLE_PARQUET)
df.printSchema()
df.show(2)
 

root
 |-- ip: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- response_bytes: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)

+------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+
|          ip|method|                 url|protocol|status_code|response_bytes|            referrer|          user_agent|    event_timestamp|
+------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+
|54.36.149.41|   GET|/filter/27|13%20%...|HTTP/1.1|        200|         30577|                   -|Mozilla/5.0 (comp...|2019-01-22 00:26:14|
| 31.56.96.51|   GET|/image/60844/prod...|HTTP/1.1|        200|

In [0]:
# bot detection

from pyspark.sql import functions as F

bot_patterns = [
    "googlebot",
    "bingbot",
    "ahrefsbot",
    "applebot",
    "yandexbot",
    "baiduspider",
    "duckduckbot",
    "semrushbot",
    "mj12bot",
    "petalbot",
    "sogou",
    "crawler",
    "spider",
    "bot"
]

regex = "|".join(bot_patterns)

df_bot = (
    df.withColumn(
        "bot_type",
        F.when(
            F.lower(F.col("user_agent")).rlike("googlebot"),
            "GoogleBot"
        )
        .when(
            F.lower(F.col("user_agent")).rlike("bingbot"),
            "BingBot"
        )
        .when(
            F.lower(F.col("user_agent")).rlike("ahrefsbot"),
            "AhrefsBot"
        )
        .when(
            F.lower(F.col("user_agent")).rlike("applebot"),
            "AppleBot"
        )
        .when(
            F.lower(F.col("user_agent")).rlike(regex),
            "OtherKnownBot"
        )
        .otherwise("Human")
    )
)

In [0]:
# bot distribution KPI
# Traffic Distribution
(
    df_bot
    .groupBy("bot_type")
    .count()
    .orderBy(F.desc("count"))
).show(truncate=False)


+-------------+-------+
|bot_type     |count  |
+-------------+-------+
|Human        |9249123|
|GoogleBot    |801797 |
|BingBot      |200700 |
|AhrefsBot    |57178  |
|OtherKnownBot|34496  |
|AppleBot     |21783  |
+-------------+-------+



In [0]:
# Bandwidth Consumed by Bots
(
    df_bot
    .groupBy("bot_type")
    .agg(
        F.sum("response_bytes").alias("bytes")
    )
    .withColumn(
        "gb",
        F.round(F.col("bytes")/(1024**3),2)
    )
    .orderBy(F.desc("gb"))
).show()

+-------------+------------+------+
|     bot_type|       bytes|    gb|
+-------------+------------+------+
|        Human|107750806321|100.35|
|    GoogleBot| 12581209714| 11.72|
|      BingBot|  5786870709|  5.39|
|    AhrefsBot|  1573264932|  1.47|
|OtherKnownBot|   609107941|  0.57|
|     AppleBot|   569736855|  0.53|
+-------------+------------+------+



In [0]:
# Bot Traffic Percentage
total = df_bot.count()

df_percentage =(
    df_bot
    .groupBy("bot_type")
    .count()
    .withColumn(
        "pct",
        F.round(F.col("count") * 100 / total, 2)
    )
)

df_percentage.show()

+-------------+-------+-----+
|     bot_type|  count|  pct|
+-------------+-------+-----+
|     AppleBot|  21783| 0.21|
|        Human|9249123|89.23|
|    GoogleBot| 801797| 7.74|
|    AhrefsBot|  57178| 0.55|
|OtherKnownBot|  34496| 0.33|
|      BingBot| 200700| 1.94|
+-------------+-------+-----+



In [0]:
# Most Crawled URLs, bu bot
# Useful for SEO

(
    df_bot
    .filter(F.col("bot_type") != "Human")
    .groupBy("url")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
).show()


+--------------------+------+
|                 url| count|
+--------------------+------+
|/static/css/font/...|177406|
|/static/images/gu...| 39915|
|/static/images/gu...| 27997|
|/static/images/gu...| 25775|
|/static/images/gu...| 17630|
|                   /| 13148|
|/static/images/am...|  5381|
|/static/images/am...|  5104|
|/static/images/am...|  4669|
|/static/css/font/...|  3208|
|                  /m|  1826|
|    /m/filter/b1,p62|  1720|
|    /m/filter/b2,p65|  1427|
|     /m/filter/b2,p3|  1297|
|         /robots.txt|  1188|
|/m/browse/mixer/%...|  1132|
|     /m/filter/b2,p6|  1131|
|/m/browse/home-ap...|  1130|
|/m/browse/refrige...|  1096|
|    /m/filter/b1,p65|  1025|
+--------------------+------+
only showing top 20 rows


In [0]:
# most accessed url by Human
(
    df_bot
    .filter(F.col("bot_type") == "Human")
    .groupBy("url")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
).show()

+--------------------+------+
|                 url| count|
+--------------------+------+
|      /settings/logo|351980|
|/site/alexaGooleA...|103686|
|/static/css/font/...|102770|
|        /favicon.ico|102069|
|/static/images/gu...| 99023|
|/static/images/gu...| 98352|
|/static/images/gu...| 98095|
|/static/images/gu...| 97692|
|/static/images/gu...| 97667|
|/static/images/am...| 88096|
|/static/images/am...| 87305|
|/static/images/am...| 86775|
|/static/images/am...| 56114|
|/amp-helper-frame...| 47075|
|/amp-helper-frame...| 44861|
|/image/%7B%7Bbask...| 36860|
|/static/bundle-bu...| 35388|
|                   /| 34432|
|/static/images/fa...| 34060|
|/static/images/th...| 33970|
+--------------------+------+
only showing top 20 rows


In [0]:
# Product analysis

from pyspark.sql import functions as F

product_df = (
    df
    .filter(
        F.col("url").rlike("^(/m)?/product/")
    )
)

print ("products count ", product_df.count())
product_df.show(5)


products count  356942
+-------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+
|           ip|method|                 url|protocol|status_code|response_bytes|            referrer|          user_agent|    event_timestamp|
+-------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+
|  91.99.72.15|   GET|/product/31893/62...|HTTP/1.1|        200|         41483|                   -|Mozilla/5.0 (Wind...|2019-01-22 00:26:17|
|207.46.13.136|   GET|      /product/10214|HTTP/1.1|        200|         39677|                   -|Mozilla/5.0 (comp...|2019-01-22 00:26:18|
|178.253.33.51|   GET|/m/product/32574/...|HTTP/1.1|        200|         20406|https://www.zanbi...|Mozilla/5.0 (Linu...|2019-01-22 00:26:19|
|  91.99.72.15|   GET|/product/10075/13...|HTTP/1.1|        200|         41725|                   -|Mozilla/5.0 (X11;...|2019

In [0]:
# Extract Product ID
product_df = (
    product_df
    .withColumn(
        "product_id",
        F.regexp_extract(
            "url",
            r"^/(?:m/)?product/(\d+)",
            1
        )
    )
)

product_df.show(5)

+-------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+----------+
|           ip|method|                 url|protocol|status_code|response_bytes|            referrer|          user_agent|    event_timestamp|product_id|
+-------------+------+--------------------+--------+-----------+--------------+--------------------+--------------------+-------------------+----------+
|  91.99.72.15|   GET|/product/31893/62...|HTTP/1.1|        200|         41483|                   -|Mozilla/5.0 (Wind...|2019-01-22 00:26:17|     31893|
|207.46.13.136|   GET|      /product/10214|HTTP/1.1|        200|         39677|                   -|Mozilla/5.0 (comp...|2019-01-22 00:26:18|     10214|
|178.253.33.51|   GET|/m/product/32574/...|HTTP/1.1|        200|         20406|https://www.zanbi...|Mozilla/5.0 (Linu...|2019-01-22 00:26:19|     32574|
|  91.99.72.15|   GET|/product/10075/13...|HTTP/1.1|        200|         41725|   

In [0]:
# Extract Product Name (Optional), persian slug, not in english, Databricks has AI translate
# not using AI now

product_df = (
    product_df
    .withColumn(
        "product_slug",
        F.regexp_extract(
            "url",
            r"^/(?:m/)?product/\d+/\d+/(.*)$",
            1
        )
    )
)

product_df.select("product_id", "product_slug").show(10)

+----------+--------------------+
|product_id|        product_slug|
+----------+--------------------+
|     31893|%D8%B3%D8%B4%D9%8...|
|     10214|                    |
|     32574|%D9%85%D8%A7%D8%B...|
|     10075|%D9%85%D8%A7%DB%8...|
|     14926|                    |
|     32798|%DB%8C%D8%AE%DA%8...|
|     33978|%DA%AF%D9%88%D8%B...|
|     30649|                    |
|      7793|%D9%85%D8%A7%DB%8...|
|     81900|                    |
+----------+--------------------+
only showing top 10 rows


In [0]:
top_products = (
    product_df
    .groupBy("product_id")
    .count()
    .orderBy(F.desc("count"))
)

top_products.show(10)

+----------+-----+
|product_id|count|
+----------+-----+
|          | 8451|
|     33953| 1538|
|     33968| 1400|
|     34286| 1369|
|     33956| 1285|
|     33960| 1257|
|     33954| 1244|
|     31720| 1166|
|     33722|  939|
|     33952|  928|
+----------+-----+
only showing top 10 rows


In [0]:
# daily popular count
top_products_daily = (
    product_df
    .groupBy(
        F.to_date("event_timestamp").alias("date"),
        "product_id"
    )
    .count()
)

top_products_daily.show(10)

+----------+----------+-----+
|      date|product_id|count|
+----------+----------+-----+
|2019-01-22|     33631|   75|
|2019-01-22|     26579|    5|
|2019-01-22|      4053|    2|
|2019-01-22|     33596|   73|
|2019-01-22|     25730|    2|
|2019-01-22|     33643|   64|
|2019-01-22|     27892|    2|
|2019-01-22|     26297|   17|
|2019-01-22|      8147|    1|
|2019-01-22|     27742|    8|
+----------+----------+-----+
only showing top 10 rows
